# Load BV-BRC data

## download all 16S genes from BV-BRC

### metadata

In [1]:
from json import dump, load
import requests

batch_size = 25000
count = 415
call = f'https://www.bv-brc.org/api/genome_feature/?and(eq(annotation,PATRIC),eq(product,"16S rRNA"))&limit({batch_size},{count*batch_size})' #&http_accept=application/dna+fasta'
request = requests.get(call)
metadata_16S = {} #load(open("datacache/16S_metadata.json", "r"))
data = request.json() #.json()
while data:
    count += 1
    for gene in data:
        genomeID = gene.pop("feature_id")
        if genomeID in metadata_16S: continue
        metadata_16S[genomeID] = gene
    if count % 5 == 0:
        dump(metadata_16S, open(f"16S_metadata{count}.json", "w"))
        metadata_16S = {}

    call = f'https://www.bv-brc.org/api/genome_feature/?and(eq(annotation,PATRIC),eq(product,"16S rRNA"))&limit({(count+1)*batch_size},{count*batch_size})' #&http_accept=application/dna+fasta"
    request = requests.get(call)
    data = request.json()
    print(count)


416
417
418
419
420


In [2]:
from json import load, dump
from glob import glob
mainMD = {}
for md in glob("16S_metadata*.json"):
    MD = load(open(md, 'r'))
    mainMD.update(MD)

dump(mainMD, open("16S_metadata.json", "w"))

### sequences

In [17]:
from json import dump, load
import requests



batch_size = 25000
count = 335
start = batch_size * count
call = f'https://www.bv-brc.org/api/genome_feature/?and(eq(annotation,PATRIC),eq(product,"16S rRNA"))&limit({batch_size},{start})&http_accept=application/dna+fasta'
request = requests.get(call)
Fasta16S = {} #load(open("Fasta16S.json", "r"))
data = request.text.strip() #.json()
while data:
    count += 1
    lines = data.split("\n")
    for line in lines:
        if line.startswith(">"):
            genomeID = line[1:]
            if genomeID in Fasta16S: continue
            Fasta16S[genomeID] = ""
        else:
            Fasta16S[genomeID] += line
    if count % 5 == 0:
        dump(Fasta16S, open(f"Fasta16S{count}.json", "w"))
        Fasta16S = {}

    call = f'https://www.bv-brc.org/api/genome_feature/?and(eq(annotation,PATRIC),eq(product,"16S rRNA"))&limit({batch_size},{count*batch_size+start})&http_accept=application/dna+fasta'
    request = requests.get(call)
    data = request.text.strip() #.json()
    # display(metadata_16S)
    # break
    print(count)


336


In [19]:
from json import load, dump
from glob import glob
mainFasta = {}
for fasta in glob("Fasta*.json"):
    fa = load(open(fasta, 'r'))
    mainFasta.update(fa)

dump(mainFasta, open("mainFasta.json", "w"))

## load predownloaded 16S genes

In [ ]:
#TODO:  this needs to be updated, possibly using the newes scrape of 16s RNA from PATRIC
gene_ids = {}
with open('data/16s_RNA_PATRIC.frn', 'r') as file:
    content = file.read()
lines = content.split('\n')
for line in lines:
    if line.startswith('>'):
        array = line.split(' ')
        gene_id = array[0][1:]
        gene_ids[gene_id] = None

util.save("16s_geneIDs", gene_ids)

## downloading 16S gene information

In [ ]:
#TODO:  this needs to be updated

import requests
from tqdm import tqdm

# genes = util.load("16s_geneIDs")
genes = util.load("16s_genes")

gene_list = []
# key_list = genes.keys()
key_list = list([g for g,v in genes.items() if v is None])
count = 0
while len(key_list) > 0:
    numLeft = len(key_list)
    for i,gene_id in tqdm(enumerate(key_list), total=len(key_list)):
        if genes[gene_id] is None:
            gene_list.append(gene_id)
        if len(gene_list) >= 100:
            query = ",".join(gene_list)
            url = f'https://www.patricbrc.org/api/genome_feature/?in(patric_id,({query}))&select(patric_id,na_sequence_md5)&http_accept=application/json'
            response = requests.get(url)
            # Check if the request was successful
            if response.status_code == 200:
                # Parse the JSON response
                data = response.json()
                for i,gene in enumerate(data):
                    if 'na_sequence_md5' in data[i]:
                        # print(data[i]['patric_id'])
                        genes[data[i]['patric_id']] = data[i]['na_sequence_md5']
                util.save("16s_genes",genes)
            else:
                count += 1
                print("Failed",count)
            gene_list = []
            # print(count, end="\r")
    if len(gene_list) > 0:
        query = ",".join(gene_list)
        url = f'https://www.patricbrc.org/api/genome_feature/?in(patric_id,({query}))&select(patric_id,na_sequence_md5)&http_accept=application/json'
        response = requests.get(url)
        # Check if the request was successful
        if response.status_code == 200:
            # Parse the JSON response
            data = response.json()
            for i,gene in enumerate(data):
                if 'na_sequence_md5' in data[i]:
                    genes[data[i]['patric_id']] = data[i]['na_sequence_md5']
            util.save("16s_genes",genes)
    key_list = list([g for g,v in genes.items() if v is None])
    if len(key_list) == numLeft:
        break

In [ ]:
md5_seqs = util.load("md5_seqs")
genes = util.load("16s_genes")
for gene_id, value in genes.items():
    if value in md5_seqs:  continue
    if value is not None:
        md5_seqs[value] = None
        # print(gene_id, value)
util.save("md5_seqs",md5_seqs)

import requests
from tqdm import tqdm

md5_seqs = util.load("md5_seqs")
failed = []
for i,md5 in tqdm(enumerate(md5_seqs), total=len(md5_seqs)):
    if md5_seqs[md5] is not None:
        continue
    url = f'https://www.patricbrc.org/api/feature_sequence/?eq(md5,({md5}))&select(sequence)&http_accept=application/json'
    response = requests.get(url)
    # Check if the request was successful
    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()
        if len(data) == 0:
            failed.append(md5)
            continue
        md5_seqs[md5] = data[0]['sequence']
        # if 
        # util.save("md5_seqs",md5_seqs)
    else:
        failed.append(md5)
    if i%100 == 0:
        util.save("md5_seqs",md5_seqs)
print(list(md5_seqs.items())[-2:])
print(f"Failed to retrieve data for {failed}")

# mapping ASVs to BV-BRC

## cutting 16S genes at the experimental primer locations

In [ ]:
from Bio import SeqUtils, SeqIO, Align, pairwise2
from numpy import mean, std
from Bio.Seq import Seq
import json
import os

amplicon_lengths = []
excessive_amplicons = []
insufficient_amplicons = []
ambiguous_basepairs = []
ambiguous_sequences = {}
def segment_16S(f_primer,r_primer,gene):
    # Find exact sequence matches with the F and then R primers.
    fwd_ePCR = SeqUtils.nt_search(str(gene),f_primer)
    rev_ePCR = SeqUtils.nt_search(str(gene),r_primer.reverse_complement())
    # If both primers match:
    if len(fwd_ePCR) < 2 or len(rev_ePCR) < 2:  return None

    print(f"Forward primer match: {fwd_ePCR}")
    print(f"Reverse primer match: {rev_ePCR}")
    # Add primer length to find start of amplicon sequence from forward match.
    start_cut = fwd_ePCR[1]+len(f_primer)
    # The beginning of the reverse primer match is the end of the amplicon.
    end_cut = rev_ePCR[1]
    # Use the start and end positions to get the eAmplicon sequence.
    eAmplicon = gene[start_cut:end_cut]
    ambiguous = str(eAmplicon).upper().count("N")
    if str(eAmplicon).upper().count("NN") > 0:
        ambiguous_sequences[gene] = ambiguous
        ambiguous_basepairs.append(ambiguous)
        return None
    eAmplicon_length = len(eAmplicon)
    if ambiguous > int(0.005*eAmplicon_length):   # floor behavior, effectively setting the limit of 0.5%
        ambiguous_sequences[gene] = ambiguous
        ambiguous_basepairs.append(ambiguous)
        return None
    
    ## filtering amplicons if their length seems unreasonable (beyond the 200-500bp range)
    if eAmplicon_length > 500:  excessive_amplicons.append(eAmplicon_length) ; return None
    elif eAmplicon_length < 200:  insufficient_amplicons.append(eAmplicon_length) ; return None
    amplicon_lengths.append(eAmplicon_length)
    print(f"eAmplicon length: {eAmplicon_length}")
    print(f"eAmplicon sequence: {eAmplicon} \n")
    
    return eAmplicon


# cut the database sequence according to the experimental primers
# forward_primer = Seq("GTGYCAGCMGCCGCGGTAA")
# reverse_primer = Seq("CCGYCAATTYMTTTRAGTTT")
forward_primer = Seq("MGCCGCGGTAA")
reverse_primer = Seq("TYMTTTRAGTTT")


Y = ["C", "T"]
M = ["A", "C"]
R = ["A", "G"]

md5_seqs = util.load("md5_seqs")
md5_cutSeqs = {}
failures = []
for md5, seq in md5_seqs.items():
    if seq is None:  failures.append(md5) ; continue
    segment = segment_16S(forward_primer,reverse_primer,seq.upper())
    if segment is None:  failures.append(md5) ; continue
    md5_cutSeqs[segment] = md5
util.save("md5_cutSeqs2", md5_cutSeqs)

print("failures,", len(failures))
print("amplicon length", mean(amplicon_lengths), "+/-", std(amplicon_lengths), "; range", min(amplicon_lengths), "-", max(amplicon_lengths))
print("excessive amplicons (>500bp):", len(excessive_amplicons), excessive_amplicons)
print("insufficient amplicons (<200bp):", len(insufficient_amplicons), insufficient_amplicons)
print("ambiguous basepairs:", mean(ambiguous_basepairs), "+/-", std(ambiguous_basepairs), "; range", min(ambiguous_basepairs), "-", max(ambiguous_basepairs))
# print("ambiguous sequences", ambiguous_sequences)

In [ ]:
lengths = [len(x.seq) for x in SeqIO.parse("/home/freiburger/Documents/sludge/data/GAME Sequencing/dna-sequences.fasta", "fasta")]
print("amplicon lengths", int(mean(lengths)), "+/-", int(std(lengths)), "; range", min(lengths), "-", max(lengths))
print(len(lengths))

## Compute the alignment of the ASVs to the BV-BRC sequences

In [ ]:
# TODO:  test these match/mismatch scoring against either synthetic data from PATRIC or from a case study
aligner = Align.PairwiseAligner(
    mode='global', match_score=1, mismatch_score=0,
    open_gap_score = -1, extend_gap_score = -0.5
    )
md5_cutSeqs2 = util.load("md5_cutSeqs2")
alignments_output_dir = "/home/freiburger/Documents/sludge/alignments/"

def alignments(amplicon):
    # print(f"{amplicon.id} is being processed.")
    output = {amplicon.id: {eAmplicon: aligner.score(amplicon, eAmplicon)/len(eAmplicon)
                            for eAmplicon in md5_cutSeqs2.keys()}}
    json.dump(output, open(f"{alignments_output_dir}/{amplicon.id}.json", "w"), indent=3)
    return output

from multiprocess import Pool
from os import cpu_count

cpus = int(cpu_count()/8)
pool = Pool(cpus)
fasta_path = "/home/freiburger/Documents/sludge/data/GAME Sequencing/dna-sequences.fasta"
args = [x for x in SeqIO.parse(fasta_path, "fasta")
        if f"{x.id}.json" not in os.listdir(alignments_output_dir)]#[:100]
print(f"{cpus} cores are being used.  The first argument is {args[0]}")
outputs = pool.map(alignments, args)

### aggregate the alignments into a single file

In [ ]:
from pandas import DataFrame
from json import load
from glob import glob
combined_outputs = {}
# for d in outputs:
for alignment in glob(f"{alignments_output_dir}/*.json"):
    d = json.load(open(alignment, 'r'))
    combined_outputs.update(d)
df = DataFrame(combined_outputs).T
df.index.name = "amplicon"
df.to_csv(alignments_output_dir + "/Midas_sludge_alignment_scores.csv", index=True)

## determining the best matches

In [ ]:
from pandas import read_csv

md5_cutSeqs2 = util.load("md5_cutSeqs2")
best_matches = {} #util.load("best_matches")
md5_geneID = {} #util.load("md5_geneID")
alignments_df = read_csv(alignments_output_dir + "/Midas_sludge_alignment_scores.csv").set_index("amplicon")
best_matches_path = "/home/freiburger/Documents/sludge/best_matches"
alignments_output_dir = "/home/freiburger/Documents/sludge/alignments/"

# display(alignments_df)
for amplicon, row in alignments_df.iterrows():
    best_matches[amplicon] = {}
    # print(f"{amplicon} is being processed.")
    # all_matches[amplicon.id] = best_match(amplicon, 0.9, aligner)
    matches = {k: round(v,2) for k,v in row.to_dict().items() if v > 0.8}
    # display(matches)
    # break
    for gene_seq, score in matches.items():
        # print(score)
        best_matches[amplicon][score] = best_matches[amplicon].setdefault(score, [])
        best_matches[amplicon][score].append(md5_cutSeqs2[gene_seq])
    best_matches[amplicon] = dict(sorted(((k, v) for k, v in best_matches[amplicon].items()), reverse=True)) # if len(v) > 3))
    # util.save(f"best_matches/{amplicon}", best_matches[amplicon])
    # break
#     filtered = dict(sorted((k, v) for k, v in matches.items() if len(v) > 3))
#     sequences = [x for x in filtered[max(filtered.keys())]]
#     json.dump({max(filtered.keys()): sequences}, open(f"best_matches/{amplicon.id}.json", "w"), indent=2)
# json.dump(all_matches, open("all_Midas_mappings.json", "w"))
util.save("best_matches", best_matches)
# best_matches

### subselect the matches to acquire

In [ ]:
genes = util.load("16s_genes")
md5_genes = {v: k for k, v in genes.items()}
util.save("md5_geneID", md5_genes)

In [ ]:
# genes = util.load("16s_genes")
md5_genes =util.load("md5_geneID")
best_matches = util.load("best_matches")
matches_to_get = {}
# output_dir = "/home/freiburger/Documents/sludge/"
for amplicon, contents in best_matches.items():
    matches_to_get[amplicon] = {}
    count = 1
    limit = False
    for score, md5s in contents.items():
        if count > 5: break
        matches_to_get[amplicon][score] = md5s
        count += len(md5s)

util.save("matches_to_get", matches_to_get)
matches_to_get

# Pulling matched genomes from BV-BRC

In [ ]:
from tqdm import tqdm
import requests

# genes = util.load("16s_genes")
md5_genes =util.load("md5_geneID")
# best_matches = util.load("best_matches")
matches_to_get = util.load("matches_to_get")
output_dir = "/home/freiburger/Documents/sludge/"
datacache_dir = "/home/freiburger/Documents/MicrobiomeNotebooks/NewWesternDiet/datacache"
for amplicon, contents in tqdm(matches_to_get.items(), total=len(matches_to_get)):
    count = 1
    limit = False
    for score, md5s in contents.items():
        for md5 in md5s:
            gID = md5_genes[md5]
            # if count > 5:
            #     limit = True
            genomeID = gID.replace("fig|", "").split(".rna")[0]
            if os.path.exists(f"{datacache_dir}/features/{genomeID}.json"):
                # count += 1
                continue
                # print(f"Already have features for {gID}")
                # continue
            # print(f"Processing {count}: {genomeID} for amplicon {amplicon} at score {score}")
            #Pull the annotations
            feat_url = (f"https://www.bv-brc.org/api/genome_feature/"
                        f"?eq(genome_id,{genomeID})"
                        "&select(patric_id,pgfam_id,figfam_id,product,feature_type,annotation)"
                        "&limit(20000)&http_accept=application/json")
            feats = requests.get(feat_url).json()
            # print(feats)
            # print(f"Genome {genomeID}: {len(feats)} features")
            # print([f for f in feats[:5]])  # show a few
            util.save(f"features/{genomeID}", feats)


            #Pull the genome FASTA
            # headers = {"Accept": "text/x-fasta"}
            # # fasta_url = (
            # #     "https://www.bv-brc.org/api/genome_sequence/"
            # #     f"?eq(genome_id,{genomeID})"
            # #     "&limit(1000000)"                    # plenty for many-contig assemblies
            # # )
            # fasta_url = f"https://www.bv-brc.org/api/genome/{genomeID}/?download&file=genome.fna"
            # genome_result = requests.get(fasta_url) #, headers=headers)
            # genome_result.raise_for_status()  # raise an error for bad responses
            # fasta_text = genome_result.text
            # print(fasta_text)


            if os.path.exists(f"{output_dir}/genomes/{genomeID}.fna"):
                # count += 1
                # print("skipping genome download")
                continue
            url = (
                "https://www.bv-brc.org/api/genome_sequence/"
                f"?eq(genome_id,{genomeID})"
                "&select(accession,sequence)"
                "&limit(100000)"
            )
            r = requests.get(url, headers={"Accept": "application/json"})
            r.raise_for_status()
            contigs = r.json()

            # Write FASTA
            with open(f"{output_dir}/genomes/{genomeID}.fna", "w") as out:
                for c in contigs:
                    out.write(f">{c['accession']}\n{c['sequence']}\n")

            # print(f"Wrote {len(contigs)} contigs to {genomeID}.fna")

            # print(fasta_text[:500])  # peek
            # open(f"{output_dir}/genomes/{genomeID}.fna", "w").write(fasta_text)
            # count += 1
        # if limit:
        #     break
    #     break
    # break
    # print(f"Finished amplicon {amplicon}")



## processing downloaded features

In [ ]:
from collections import Counter
from glob import glob
from json import load
from numpy import mean

matches_to_get = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/NewWesternDiet/datacache/matches_to_get.json", "r"))
md5_genes = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/NewWesternDiet/datacache/md5_geneID.json", "r"))
# feature_attribute = "figfam_id"
# feature_attribute = "pgfam_id"
feature_attribute = "product"
probabilities = {}
datacache_dir = "/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/NewWesternDiet/datacache"
# for f in glob("/home/freiburger/Documents/MicrobiomeNotebooks/NewWesternDiet/datacache/features/*.json"):
no_features = []
errors = []
maxes = []
for amplicon, contents in matches_to_get.items():
    probabilities[amplicon] = []
    features = []
    hits = 0
    for score, md5s in contents.items():
        hits += len(md5s)
        for md5 in md5s:
            genomeID = md5_genes[md5].replace("fig|", "").split(".rna")[0]
            with open(f"{datacache_dir}/features/{genomeID}.json") as json_file:
                try:
                    feats = set([f.get(feature_attribute, None) for f in load(json_file)])
                    feats.remove(None)
                    features.extend(list(feats))
                except Exception as e:
                    errors.append((genomeID, str(e)))
                    print(f"Error loading features for {genomeID}: {e}")

    # print(features)
    counted = Counter(features)
    # print(list(counted.values())[:5])
    probabilities[amplicon] = {k:v/hits for k,v in counted.items()}
    if len(probabilities[amplicon]) == 0:
        no_features.append(amplicon)
        continue
    # print(list(probabilities[amplicon].values())[:5])
    maxes.append(max(probabilities[amplicon].values()))
    print(amplicon, hits, len(features), max(probabilities[amplicon].values()), min(probabilities[amplicon].values()))
    # test = probabilities[amplicon]
    # break

print(feature_attribute, "mean max probability", mean(maxes))
print("no features found for", no_features)
print("errors found for", errors)
util.save("feature_probabilities", probabilities)

### assigning and visualizing gene probabilities

In [ ]:
from numpy import array, where
from statistics import median, mean
from json import load

# display(list(feature_probabilities.items())[:5])
# for asv, prob in feature_probabilities.items():
#     print(asv, prob)
#     print(max(prob.values()))

iterativeIDs = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/digestor/iterativeIDs.json", 'r'))
feature_probabilities = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/data/feature_probabilities.json", "r"))
genomes = ['Proteiniphilum.5', 'Proteiniphilum.4', 'Pseudomonas.1', 'Christensenellaceae_R-7_group.11', 'Thermovirga.4', 'Christensenellaceae_R-7_group.10', 'midas_g_36215.2', 'midas_g_12.1', 'midas_g_36215.3', 'Azoarcus.2', 'Alcaligenes.2', 'JGI-0000079-D21.1', 'Desulfovibrio.1', 'Leucobacter.1', 'Trichloromonas.1', 'midas_g_91392.1', 'Brevundimonas.2', 'Brevundimonas.3', 'Methanobacterium.4', 'Methanobacterium.5', 'Lentimicrobium.7', 'Pir4_lineage.1', 'Lentimicrobium.6', 'Methanobacterium.17', 'Methanobacterium.16', 'Ca_Cloacimonas.1', 'Pseudarcobacter.1', 'midas_g_94288.1', 'Sedimentibacter.3', 'Sedimentibacter.2', 'Lachnoclostridium.1', 'midas_g_98017.1', 'Bacteroides.2', 'Rhodobacter.1', 'Desulfosporosinus.1', 'Enterococcus.1', 'midas_g_25828.2', 'Methanothrix.1', 'midas_g_156.1', 'midas_g_110683.1', 'midas_g_75908.1', 'Aminivibrio.2', 'Alkaliphilus_oremlandii_OhILAs.1', 'midas_g_4325.2', 'Rhodococcus.1', 'Acinetobacter.1', 'Intestinibacter.1', 'Anaerosalibacter.2', 'Terrisporobacter.2', 'Dehalobacter.1', 'Syntrophomonas.1', 'Mangroviflexus.1', 'Sphingopyxis.5', 'Sphingopyxis.4', 'Enhydrobacter.1', 'midas_g_61343.1', 'LNR_A2-18.3', 'LNR_A2-18.2', 'Christensenellaceae_R-7_group.4', 'Christensenellaceae_R-7_group.5', 'Methanomassiliicoccus.1', 'midas_g_77817.1', 'Paraclostridium.1', 'midas_g_109927.1', 'Methanosarcina.2', 'Methanosarcina.3', 'Anaerovorax.1', 'midas_g_399.3', 'midas_g_399.2', 'midas_g_75.1', 'midas_g_116109.1', 'midas_g_2178.2', 'midas_g_249.1', 'Bacillus.6', 'Acetobacterium.2', 'Acetobacterium.3', 'Aquamicrobium.3', 'midas_g_6187.2', 'midas_g_6187.3', 'Aquamicrobium.2', 'midas_g_1251.1', 'Bosea.1', 'Clostridium_sensu_stricto_13.4', 'midas_g_669.1', 'Petrimonas.4', 'EBM-39.2', 'EBM-39.3', 'midas_g_7.1', 'Anaerocolumna.2', 'midas_g_36376.1', 'midas_g_93469.1', 'Sphaerochaeta.1', 'Pseudaminobacter.1', 'Angelakisella.1', 'Sphingopyxis.2', 'Sphingopyxis.3', 'Parvibaculum.1', 'Pannonibacter.1', 'Desulfomicrobium.1', 'Comamonas.1', 'midas_g_109803.1', 'Fonticella.1', 'Christensenellaceae_R-7_group.3', 'Christensenellaceae_R-7_group.2', 'midas_g_112829.1', 'midas_g_3595.1', 'midas_g_2296.1', 'Dysgonomonas.1', 'midas_g_119665.1', 'Camelimonas.2', 'Dechlorobacter.1', 'Syntrophobacter.1', 'Methanosarcina.4', 'Anaerovorax.6', 'Anaerovorax.7', 'Bacillus.1', 'midas_g_31962.1', 'Desulfobulbus.1', 'Pelotomaculum.1', 'Intestinimonas.1', 'Afipia.1', 'Christensenellaceae_R-7_group.9', 'Erysipelothrix.1', 'Christensenellaceae_R-7_group.8', 'midas_g_10303.1', 'Thiopseudomonas.1', 'Fastidiosipila.1', 'midas_g_6187.4', 'Petrimonas.3', 'midas_g_97954.1', 'Petrimonas.2', 'Clostridium_sensu_stricto_13.3', 'Ca_Ovatusbacter.1', 'Clostridium_sensu_stricto_13.2', 'Thauera.1', 'Dethiobacter.2', 'HN-HF0106.1', 'Hydrogenophaga.2', 'Sulfurospirillum.2', 'Christensenellaceae_R-7_group.16', 'Thermovirga.3', 'Christensenellaceae_R-7_group.17', 'Thermovirga.2', 'Corynebacterium.1', 'midas_g_36215.5', 'midas_g_36215.4', 'midas_g_101467.1', 'Proteiniphilum.2', 'Proteiniphilum.3', 'Proteiniclasticum.1', 'Tissierella.2', 'Desulfovibrio.6', 'Desulfovibrio.7', 'midas_g_1799.2', 'Fermentimonas.2', 'Fermentimonas.3', 'midas_g_3409.2', 'Ruminiclostridium.2', 'Lentimicrobium.1', 'midas_g_8129.1', 'Methanobacterium.10', 'Aminiphilus.1', 'Methanobacterium.11', 'midas_g_90293.1', 'Ca_Phosphitivorax.1', 'midas_g_2901.1', 'UCG-009.2', 'midas_g_4042.1', 'midas_g_1220.1', 'Methanobacterium.3', 'Methanobacterium.2', 'Monoglobus.1', 'Sedimentibacter.4', 'Achromobacter.2', 'Sedimentibacter.5', 'Achromobacter.3', 'Colidextribacter.1', 'Romboutsia.1', 'midas_g_4301.2', 'Clostridium_sensu_stricto_1.1', 'Acetoanaerobium.1', 'midas_g_9269.2', 'midas_g_9269.3', 'Oscillibacter.2', 'Caproiciproducens.3', 'Caproiciproducens.2', 'Syner-01.1', 'midas_g_343.1', 'Pseudochelatococcus.1', 'Methanobacterium.9', 'Methanobacterium.8', 'midas_g_497.1', 'Smithella.1', 'Microbacterium.1', 'Stenotrophomonas.1', 'Ochrobactrum.2', 'Gordonia.2', 'Gordonia.3', 'Mycobacterium.2', 'Anaerostignum.1', 'Methanosarcina.1', 'Anaerovorax.2', 'Anaerovorax.3', 'midas_g_77817.2', 'midas_g_399.1', 'Tuzzerella.1', 'midas_g_2178.1', 'Syntrophorhabdus.1', 'midas_g_114295.1', 'Mesotoga.1', 'Bacillus.4', 'Bacillus.5', 'midas_g_6187.1', 'Aquamicrobium.1', 'midas_g_93128.1', 'Acetobacterium.1', 'Bosea.2', 'Desulfurispora.1', 'Erysipelothrix.4', 'Leptolinea.1', 'RBG-16-49-21.1', 'EBM-39.1', 'Anaerocolumna.1', 'midas_g_52.1', 'midas_g_669.2', 'midas_g_669.3', 'Chryseobacterium.1', 'midas_g_75908.3', 'midas_g_75908.2', 'DMER64.1', 'Epulopiscium.1', 'midas_g_5426.1', 'midas_g_4325.1', 'Aminivibrio.1', 'midas_g_92370.1', 'Sphaerochaeta.4', 'Acinetobacter.2', 'Acinetobacter.3', 'Terrisporobacter.1', 'Anaerosalibacter.1', 'Actinomyces.1', 'LNR_A2-18.1', 'Christensenellaceae_R-7_group.7', 'Christensenellaceae_R-7_group.6', 'Thermobrachium.1', 'Ca_Cloacimonas.2', 'Methanobacterium.20', 'Sedimentibacter.1', 'midas_g_30443.1', 'midas_g_94288.2', 'Lachnoclostridium.3', 'Lachnoclostridium.2', 'midas_g_98017.2', 'Syner-01.4', 'Christensenellaceae_R-7_group.18', 'Syner-01.5', 'midas_g_33659.1', 'midas_g_765.1', 'Clostridium_sensu_stricto_11.1', 'midas_g_114992.1', 'midas_g_107376.1', 'Ca_Soleaferrea.1', 'Bacteroides.1', 'Methanoculleus.1', 'midas_g_25828.1', 'Enterococcus.3', 'Soehngenia.1', 'Enterococcus.2', 'Clostridium_sensu_stricto_7.1', 'Stenotrophomonas.4', 'midas_g_2455.1', 'midas_g_515.1', 'midas_g_4387.1', 'Christensenellaceae_R-7_group.12', 'Christensenellaceae_R-7_group.13', 'midas_g_36215.1', 'Pseudomonas.3', 'Pseudomonas.2', 'midas_g_94069.1', 'Alcaligenes.1', 'Azoarcus.1', 'Ercella.1', 'Dehalobacterium.1', 'midas_g_82137.1', 'Desulfitibacter.1', 'Desulfovibrio.2', 'Desulfovibrio.3', 'Methanobacterium.7', 'Methanobacterium.6', 'midas_g_91392.2', 'Brevundimonas.1', 'Andreesenia_angusta.1', 'Methanobacterium.14', 'Methanobacterium.15', 'Lentimicrobium.4', 'Lentimicrobium.5', 'Achromobacter.1', 'midas_g_5574.1', 'Sedimentibacter.7', 'Sedimentibacter.6', 'Romboutsia.3', 'Colidextribacter.2', 'Romboutsia.2', 'Colidextribacter.3', 'Monoglobus.2', 'midas_g_9023.1', 'Acetoanaerobium.2', 'midas_g_9269.1', 'Oscillibacter.1', 'midas_g_4301.1', 'Clostridium_sensu_stricto_1.2', 'Caproiciproducens.1', 'Lachnoclostridium.4', 'Syner-01.3', 'Syner-01.2', 'midas_g_102931.1', 'Enterococcus.4', 'Rummeliibacillus.1', 'Lentimicrobium.9', 'Lentimicrobium.8', 'Methanobacterium.19', 'Methanobacterium.18', 'Stenotrophomonas.2', 'Stenotrophomonas.3', 'Ochrobactrum.1', 'midas_g_19.1', 'Hyphomonas.1', 'Mycobacterium.1', 'Gordonia.1', 'Sporanaerobacter.1', 'Amaricoccus.1', 'Christensenellaceae_R-7_group.15', 'Thermovirga.1', 'Christensenellaceae_R-7_group.14', 'Pseudomonas.4', 'Pseudomonas.5', 'Hydrogenophaga.1', 'Sulfurospirillum.1', 'Proteiniphilum.1', 'midas_g_115459.1', 'midas_g_21833.1', 'Levilinea.1', 'midas_g_1799.1', 'Fermentimonas.1', 'Tissierella.1', 'midas_g_83754.1', 'Desulfovibrio.5', 'Desulfovibrio.4', 'midas_g_112760.1', 'midas_g_3409.1', 'Ruminiclostridium.1', 'Methanobacterium.13', 'Methanobacterium.12', 'midas_g_114252.1', 'Lentimicrobium.3', 'Lentimicrobium.2', 'IMCC26207.1', 'Methanobacterium.1', 'UCG-009.1', 'Dysgonomonas.2', 'midas_g_49937.1', 'Anaerovorax.5', 'Turicibacter.1', 'Anaerovorax.4', 'Camelimonas.1', 'Clostridium_sensu_stricto_5.1', 'midas_g_76594.1', 'Bacillus.3', 'Bacillus.2', 'Thiobacillus.1', 'Proteiniborus.1', 'Pusillimonas.1', 'midas_g_531.1', 'Pelotomaculum.2', 'Desulfobulbus.2', 'Desulfobulbus.3', 'midas_g_112372.1', 'Erysipelothrix.2', 'Erysipelothrix.3', 'Ca_Caldatribacterium.1', 'Brooklawnia.1', 'midas_g_15299.1', 'midas_g_97954.3', 'midas_g_669.5', 'Petrimonas.1', 'midas_g_97954.2', 'midas_g_669.4', 'Dethiobacter.1', 'Clostridium_sensu_stricto_13.1', 'midas_g_75908.4', 'midas_g_269.1', 'midas_g_100.1', 'Taibaiella.1', 'midas_g_2022.1', 'Sphaerochaeta.3', 'Sphaerochaeta.2', 'Stappia.1', 'midas_g_1981.1', 'Sphingopyxis.1', 'NK4A214_group.1', 'midas_g_93666.1', 'Fonticella.2', 'Christensenellaceae_R-7_group.1']

ASV_gene_probs = {}
missing = []
for asv_md5, prob in feature_probabilities.items():
    # print(gene, prob)
    asv = iterativeIDs.get(asv_md5, None)
    if asv is None:
        missing.append(asv_md5)
        continue
    # splits = gene.split("_")
    # if len(splits) > 2:
    #     geneCount = splits[-1]
    #     asv = "_".join(splits[:-1])
    # else:
    #     asv, geneCount = gene.split("_")
    ASV_gene_probs.setdefault(asv, {})
    ASV_gene_probs[asv].update(prob)

print("undefined iterative ASVs", missing)

# print(max([max(ps) for ps in ASV_gene_probs.values()]))
probs = {asv: array(list(content.values())) #/ sum(list(content.values()))
                for asv, content in ASV_gene_probs.items()}
# print(max([max(ps) for ps in probs.values()]))
probs = {asv: where(prob < 1e-5, 0, prob) for asv, prob in probs.items()}

# display(list(ASV_gene_probs.keys()))
probabilities = []
# missing = []
for asv, prob in probs.items():
    # print(max(prob))
    probabilities.extend(list(prob))
probabilities = array(probabilities)
print(len(probabilities[probabilities < 0.5])/len(probabilities))

missing = set(genomes).symmetric_difference(set(probs.keys()))
print("missing", len(missing), missing)

from numpy import mean, std 
print("max:", max(probabilities), "min:", min(probabilities),
        "mean:", mean(probabilities), "+/-", std(probabilities))

from matplotlib import pyplot as plt
plt.figure(figsize=(10, 6))
counts, bins, patches = plt.hist(probabilities, bins=30, edgecolor='black', color='skyblue', alpha=0.7)

# Coordinates: (x, y) in axes fraction (0=left/bottom, 1=right/top)
ax = plt.gca()
title_font = 20
axis_font = 14
plt.text(0.9, 0.9,
    f'Median: {round(median(probabilities), 2)}\nMean: {round(mean(probabilities), 2)}',
    transform=ax.transAxes,
    fontsize=axis_font,
    # fontweight='bold',
    # color='darkred',
    horizontalalignment='right',     # align text to the right
    verticalalignment='top',         # align text to the top
    bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat", alpha=0.8)
)

# Styling
plt.title('Probability Distribution of ASV Genes', fontsize=title_font)
plt.xlabel('Gene Probability', fontsize=16)
plt.ylabel('# of Genes', fontsize=16)
plt.xticks(fontsize=axis_font)
plt.yticks(fontsize=axis_font)
plt.yscale('log')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("feature_probabilities.png")
plt.show()